# Kaggle Dataset Preparation

This notebook prepares the Kaggle Human-LLM Generated Phishing/Legitimate Emails dataset for binary text-only phishing detection. It covers dataset inspection, unified text field construction across two source schemas, duplicate handling, and the final 60/40 train-test split.

## 1. Import libraries and load the raw dataset

This section imports the required libraries and loads the raw Kaggle CSV file for initial inspection.

In [1]:
# Import pathlib for working with file paths in a clean way
from pathlib import Path

# Import os for listing files in a folder
import os

# Import pandas for loading and working with tabular data
import pandas as pd

In [2]:
# Set the path to the folder where the Kaggle raw data was downloaded
data_path = Path("../data/raw/kaggle_human_llm/")

# Walk through all subfolders and print every file found
for folder in sorted(data_path.iterdir()):
    # iterdir() returns all items inside data_path (files and folders)
    if folder.is_dir():
        print(f"{folder.name}/")
        for file in sorted(folder.iterdir()):
            print(f"  {file.name}")

human-generated/
  legit.csv
  phishing.csv
llm-generated/
  legit.csv
  phishing.csv


In [3]:
frames = []

# Loop through each source folder and each file, assigning the correct binary label
for source_folder in ["human-generated", "llm-generated"]:
    for filename, label in [("phishing.csv", 1), ("legit.csv", 0)]:
        filepath = data_path / source_folder / filename

        # Load the CSV file into a temporary DataFrame
        temp_df = pd.read_csv(filepath, sep=None, engine="python", on_bad_lines="skip")

        # Assign label: 1 = phishing, 0 = legitimate
        temp_df["label"] = label

        # Add a column to track which folder this row came from
        temp_df["source"] = source_folder
        
        frames.append(temp_df)

# Stack all four DataFrames into one
df = pd.concat(frames, ignore_index=True)

print("Combined shape:", df.shape)
df.head(3)

Combined shape: (3595, 9)


,sender,receiver,date,subject,body,urls,label,source,text
0,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1,human-generated,NaN
1,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1,human-generated,NaN
2,Fastway\n\t<info.fastway.co.za_info.fastway.co...,jose@monkey.org,2022-12-21 01:33:32,Your shipment is on the way,Announcing JotForm Tables: When a spreadsheet ...,1,1,human-generated,NaN


## 2. Inspect schema and content

This section examines the dataset's columns, data types, and example rows to understand the structure before any preprocessing decisions are made.

In [4]:
# Print a list of all column names in the dataset
df.columns.tolist()

['sender',
 'receiver',
 'date',
 'subject',
 'body',
 'urls',
 'label',
 'source',
 'text']

In [5]:
# Display the first 5 rows to see what the data looks like
df.head(5)

,sender,receiver,date,subject,body,urls,label,source,text
0,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1,human-generated,NaN
1,MetaMask <info@sofamekar.com>,jose@monkey.org,2022-12-27 10:56:49,Your MetaMask wallet will be suspended,Verify your MetaMask Wallet Our system has sho...,1,1,human-generated,NaN
2,Fastway\n\t<info.fastway.co.za_info.fastway.co...,jose@monkey.org,2022-12-21 01:33:32,Your shipment is on the way,Announcing JotForm Tables: When a spreadsheet ...,1,1,human-generated,NaN
3,Fastway\n\t<info.fastway.co.za_info.fastway.co...,jose@monkey.org,2022-12-21 01:33:32,Your shipment is on the way,Announcing JotForm Tables: When a spreadsheet ...,1,1,human-generated,NaN
4,Help Center\n\t<info.help-center.co.za_info.he...,jose@monkey.org,2022-12-20 23:00:27,Netflix : We're having some trouble with your ...,"HELLO, Please note that, your monthly paymen...",1,1,human-generated,NaN


In [6]:
# Show each column's data type and how many non-null values it has
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 3595 entries, 0 to 3594
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   sender    2000 non-null   str   
 1   receiver  1962 non-null   str   
 2   date      2000 non-null   str   
 3   subject   1998 non-null   str   
 4   body      2000 non-null   str   
 5   urls      2000 non-null   object
 6   label     3595 non-null   int64 
 7   source    3595 non-null   str   
 8   text      1355 non-null   str   
dtypes: int64(1), object(1), str(7)
memory usage: 6.2+ MB


In [7]:
# Display 5 randomly selected rows using a fixed seed so the same rows appear each time
df.sample(5, random_state=42)

,sender,receiver,date,subject,body,urls,label,source,text
691,Email Database < hr@okrabooking.com >,jose@monkey.org,2020-07-15 08:23:35,Suspension Notice- ACTION REQUIRED!,Access to jose@monkey.org will be suspended a...,1,1,human-generated,NaN
1672,Joe Sloan <njk@tmsusa.com>,wkilxloc@opensuse.org,2008-08-05 16:18:32,Re: [opensuse] Intel 3945 Wireless Problem ope...,Chuck Stuettgen wrote:\r\n\r\n> The fix is to ...,0,0,human-generated,NaN
721,"""=?utf-8?B?Tm90aWNl?="" <contact@jmcpavi.com.br>","""jose@monkey.org"" <jose@monkey.org>",2020-06-18 00:30:09,=?utf-8?B?WW91ciBOZXRmbGl4IE1lbWJlcnNoaXAgaXMg...,92B57697-F05D-41B8-A7B2-093A7D29529C 0468FB78...,1,1,human-generated,NaN
3249,NaN,NaN,NaN,NaN,NaN,NaN,0,llm-generated,"Dear Ms. Amanda Thompson, I hope this email fi..."
26,"""monkey.org Mail Center"" <jose@genfiak.host>",jose@monkey.org,2022-11-30 02:19:01,You have secured emails in monkey.org mailserver,domain555.html You have 7 secure messages in t...,1,1,human-generated,NaN


## 3. Examine data quality and structure

This section checks missing values and column variability to identify any quality issues before cleaning.

In [8]:
# Count missing values in each column, sorted so the worst columns appear first
df.isna().sum().sort_values(ascending=False)

text        2240
receiver    1633
subject     1597
sender      1595
date        1595
body        1595
urls        1595
label          0
source         0
dtype: int64

In [9]:
# For each column, print how many unique values it contains (including NaN)
for col in df.columns:
    print(col, df[col].nunique(dropna=False))

sender 819
receiver 257
date 1199
subject 979
body 1231
urls 146
label 2
source 2
text 1163


## 4. Inspect label distribution

This dataset contains four classes: human-written phishing, LLM-generated phishing, human-written legitimate, and LLM-generated legitimate. This section confirms the class counts before collapsing to a binary label for consistent evaluation with MeAJOR.

In [10]:
# Count how many emails belong to each of the four original classes
print(df["label"].value_counts())
print()

# Show those counts as proportions of the total
print(df["label"].value_counts(normalize=True))

label
0    2000
1    1595
Name: count, dtype: int64

label
0    0.556328
1    0.443672
Name: proportion, dtype: float64


## 5. Collapse to binary labels

For consistent binary classification across both datasets, all phishing emails (human and LLM-generated) are assigned label 1 and all legitimate emails are assigned label 0. This collapses the four original classes into two.

In [11]:
# Confirm labels are already binary, no collapsing needed since labels were assigned during loading
print("Unique label values:", df["label"].unique())
print("Label counts:\n", df["label"].value_counts())

Unique label values: [1 0]
Label counts:
 label
0    2000
1    1595
Name: count, dtype: int64


## 6. Build the final input text

This section selects the relevant text fields and combines them into a single input field. Subject and body are joined where both are present, otherwise only the available field is used.

In [12]:
# Create a boolean mask, True for human-generated rows, False for llm-generated rows
human_mask = df["source"] == "human-generated"

# For human rows: join subject and body into one string
# fillna("") stops NaN values turning into the string "nan"
df.loc[human_mask, "text"] = (
    df.loc[human_mask, "subject"].fillna("").str.strip()
    + " "
    + df.loc[human_mask, "body"].fillna("").str.strip()
).str.strip()

# If both subject and body were NaN, the result is an empty string, convert to NaN
df["text"] = df["text"].replace("", pd.NA)

# LLM rows already have a text column populated, leave them untouched

# Confirm nulls remaining, these will be dropped in Section 8
print("Null text values remaining:", df["text"].isna().sum())
df[["text", "label", "source"]].head(3)

Null text values remaining: 240


,text,label,source
0,Your MetaMask wallet will be suspended Verify ...,1,human-generated
1,Your MetaMask wallet will be suspended Verify ...,1,human-generated
2,Your shipment is on the way Announcing JotForm...,1,human-generated


## 7. Check and remove duplicates

This section checks for duplicate text-label pairs and removes them to prevent repeated examples from inflating model performance.

In [13]:
# Count how many rows have identical text to an earlier row
num_dupes = df["text"].duplicated().sum()
print(f"Duplicate text rows found: {num_dupes}")

# drop_duplicates keeps the first occurrence and removes all later copies
# subset="text" means only the text column is used to identify duplicates
df = df.drop_duplicates(subset="text").reset_index(drop=True)

print(f"Shape after removing duplicates: {df.shape}")

Duplicate text rows found: 1202
Shape after removing duplicates: (2393, 9)


## 8. Remove empty text rows

This section removes any rows where the final text field is empty after preprocessing.

In [14]:
# Replace any whitespace-only strings with NaN so dropna can catch them
df["text"] = df["text"].replace("", pd.NA)

# Drop rows where text is NaN — they contain no usable content for training
df = df.dropna(subset=["text"]).reset_index(drop=True)

print(f"Shape after removing empty text rows: {df.shape}")

# Confirm zero nulls remain in the text column
print(f"Null text values remaining: {df['text'].isna().sum()}")

Shape after removing empty text rows: (2392, 9)
Null text values remaining: 0


## 9. Create the 60/40 train-test split

A stratified 60/40 train-test split is applied using a fixed random seed of 42, consistent with the MeAJOR dataset split. Stratification ensures both partitions preserve the binary class balance.

In [15]:
# Import the train_test_split utility from scikit-learn
from sklearn.model_selection import train_test_split

# Keep only the columns needed for modelling plus source for evaluation
df_clean = df[["text", "label", "source"]].copy()

# test_size=0.4 puts 40% into test and keeps 60% for training
# stratify ensures both splits have the same phishing/legitimate ratio as the full dataset
# random_state=42 makes the split reproducible
train_df, test_df = train_test_split(
    df_clean,
    test_size=0.4,
    random_state=42,
    stratify=df_clean["label"]
)

print(f"Train size: {len(train_df)} rows")
print(f"Test size:  {len(test_df)} rows")
print(f"\nTrain label distribution:\n{train_df['label'].value_counts()}")
print(f"\nTest label distribution:\n{test_df['label'].value_counts()}")

Train size: 1435 rows
Test size:  957 rows

Train label distribution:
label
0    1034
1     401
Name: count, dtype: int64

Test label distribution:
label
0    690
1    267
Name: count, dtype: int64


## 10. Preparation summary and save outputs

The Kaggle dataset has been reduced to a binary text-only format: phishing (label 1) and legitimate (label 0). All four original classes have been preserved within this binary structure, including both human-written and LLM-generated emails. The cleaned dataset has been split into reproducible 60/40 train and test sets and saved to the processed data folder ready for baseline modelling.

In [17]:
# Print a final summary of the processed dataset
print("=== Kaggle Dataset Preparation Summary ===")
print(f"Total rows after cleaning:     {len(df_clean)}")
print(f"Training rows (60%):           {len(train_df)}")
print(f"Test rows (40%):               {len(test_df)}")
print(f"Label balance (full dataset):\n{df_clean['label'].value_counts()}")
print(f"\nSource breakdown:\n{df_clean['source'].value_counts()}")
print()

# Build the path to the output folder
output_dir = Path("../data/processed/kaggle/")

# exist_ok=True prevents an error if the folder already exists
output_dir.mkdir(parents=True, exist_ok=True)

# Save the full cleaned dataset
df_clean.to_parquet(output_dir / "kaggle_text_label_dedup.parquet", index=False)

# Save the training split
train_df.to_parquet(output_dir / "kaggle_train_60.parquet", index=False)

# Save the test split
test_df.to_parquet(output_dir / "kaggle_test_40.parquet", index=False)

print("Files saved:")
for f in sorted(output_dir.iterdir()):
    # st_size returns file size in bytes, divide by 1024 to show kilobytes
    print(f"  {f.name}  ({f.stat().st_size / 1024:.1f} KB)")
print(f"\nSaved to: {output_dir.resolve()}")

=== Kaggle Dataset Preparation Summary ===
Total rows after cleaning:     2392
Training rows (60%):           1435
Test rows (40%):               957
Label balance (full dataset):
label
0    1724
1     668
Name: count, dtype: int64

Source breakdown:
source
human-generated    1230
llm-generated      1162
Name: count, dtype: int64

Files saved:
  kaggle_test_40.parquet  (668.1 KB)
  kaggle_text_label_dedup.parquet  (1473.2 KB)
  kaggle_train_60.parquet  (1146.7 KB)

Saved to: /Users/ziliang/Downloads/cm3203-phishing-nlp/data/processed/kaggle
